# VDSR ×3 — 깊게 쌓고 차이만 배우기

흐린 위성사진(10 m) → 3배 선명하게(3.33 m).

SRCNN 과 규약이 같다. 먼저 bicubic 으로 3배 키운 뒤 그 흐린 영상을 다듬고,
밝기(Y) 채널 하나만 본다. 다른 점은 둘이다.

- **conv 3장 → 20장.** 훨씬 깊다.
- **잔차 학습.** 20층이 정답과 입력의 *차이* 만 배우고 마지막에 입력을 더한다.

차이만 배우면 학습 시작부터 출력이 bicubic 수준이라, 높은 학습률(0.1)을
써도 된다. 대신 터지지 않게 기울기를 자른다(0.01).

| | 구조 | 손실 | 파라미터 |
|---|---|---|---|
| SRCNN | conv 3장 | MSE | 57.3K |
| **VDSR** | **conv 20장 + 잔차** | **MSE** | **0.66M** |

## 1. 데이터

In [ ]:
import sys, urllib.request

LIB = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main/lib'
for m in ['sr_utils.py', 'vdsr_arch.py', 'vdsr_imgproc.py', 'vdsr_models.py']:
    urllib.request.urlretrieve(f'{LIB}/{m}', m)
    sys.modules.pop(m[:-3], None)     # 이미 불러온 옛 모듈이 남아 있으면 비운다

from sr_utils import *

show_data()        # validation 2패치 + test 2구역

## 2. 훈련

**코드가 도는지 확인하는 용도다.** 16장으로 1 epoch 만 돌린다.
아래 결과는 전체 데이터로 학습해둔 가중치를 쓴다.

업스트림 설정 그대로다 — SGD 학습률 0.1, 기울기 자르기 0.01.
잔차 학습이 아니면 이 학습률에서 바로 발산한다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from vdsr_models import build_vdsr, vdsr_pairs

N_TRAIN, EPOCHS, BATCH, CLIP = 16, 1, 4, 0.01
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

lo, hi = zip(*[pair('training', s) for s in list_split('training')[:N_TRAIN]])
x, t = vdsr_pairs(lo, hi)         # 입력: bicubic 으로 키운 LR 의 Y / 목표: HR 의 Y
loader = DataLoader(TensorDataset(x, t), batch_size=BATCH, shuffle=True)

net = build_vdsr().to(dev).train()
opt = torch.optim.SGD(net.parameters(), 0.1, momentum=0.9, weight_decay=1e-4)
crit = nn.MSELoss()

print(f'VDSR  {sum(p.numel() for p in net.parameters())/1e6:.2f}M')
for ep in range(1, EPOCHS + 1):
    tot = 0.0
    for xb, tb in loader:
        loss = crit(net(xb.to(dev)), tb.to(dev))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), CLIP)
        opt.step()
        tot += loss.item()
    print(f'epoch {ep}/{EPOCHS}   MSE {tot/len(loader):.6f}')

## 3. 학습 로그

전체 데이터로 돌린 기록이다. 학습률이 12 epoch 마다 1/10 로 떨어진다.

마지막 2 epoch 은 **g_LR 변형 10종을 합친 학습셋**으로 이어 학습한 구간이다
(같은 HR 에 열화 세기가 다른 LR 10벌). 세로 점선이 그 경계다.

In [ ]:
import pandas as pd

MODEL = f'{BASE}/models/02_vdsr_x3'
e = pd.read_csv(fetch(f'{MODEL}/statistics/train_results.csv', 'log.csv'), index_col=0)
# 앞 30 epoch + 마지막의 '변형 10종' 추가 학습 구간
e = pd.concat([e[e.phase == 1].head(30), e[e.phase == 2]])
cut = e[e.phase == 1].index.max()      # 데이터를 바꾼 지점

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(e.index, e.mse, color='#2f6f9f', lw=1.6)
ax[0].set_yscale('log'); ax[0].set_title('Training MSE loss'); ax[0].set_ylabel('MSE')

ax[1].plot(e.index, e.PSNR, color='#4f9d69', lw=1.6)
ax[1].set_title('Validation PSNR during training'); ax[1].set_ylabel('dB')

ax[2].step(e.index, e.lr, color='#c96a5b', lw=1.6, where='post')
ax[2].set_yscale('log'); ax[2].set_title('Learning rate (StepLR)')
for a in ax:
    a.axvline(cut, color='#888', ls='--', lw=1.2)   # 데이터를 바꾼 지점
    a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

print(f'MSE  {e.mse.iloc[0]:.6f} -> {e.mse.iloc[-1]:.6f}   ({len(e)} epoch)')

## 4. 결과

In [ ]:
from vdsr_models import load_vdsr, vdsr_upscale

net = load_vdsr(fetch(f'{MODEL}/checkpoints/vdsr_x3.pth', 'vdsr_x3.pth'))
print(f'VDSR  {sum(p.numel() for p in net.parameters())/1e6:.2f}M')

upscale = lambda lr: vdsr_upscale(net, lr)

show_results(upscale, 'VDSR', center=[(137, 236), (313, 346)])

## 5. 평가

In [ ]:
rows = compare(upscale, label='VDSR')

## 6. 최종 테스트 — 인천

정답이 없는 실제 Sentinel-2 촬영본이다. 점수는 못 내고 눈으로 확인한다.

In [ ]:
show_test(upscale, 'VDSR', center=(800, 800), size=70)